### Reducible Word

- 時間複雜度：$O(N^2 \cdot K)$
  - $K$：代表所有可能的合法子單字數量。因為有 memo，每個單字只會被完整「展開」檢查一次。
  - $N$：目標單字的長度。
  - 推導：
    - 在每個單字的 is_reducible 呼叫中，我們會跑一個 $O(N)$ 的迴圈。
    - 在迴圈內，字串切片 word[:i] + word[i+1:] 也是 $O(N)$。
    - 因此每個狀態的處理成本是 $O(N^2)$。
    - 為了匹配，我們最少要遞回 N 次，但中間可能會有失敗，所以總遞回次數是所有 memo 記錄的狀態次數 K 。
- $O(M + K \cdot N + N)$
  - $M$ (輸入資料空間)：valid_words 集合所佔用的空間。這是必須存在的輸入成本。
  - $K \cdot N$ (輔助快取空間)：self.memo 字典。
    - $K$：被記錄在 memo 中的單字總數。
    - $N$：單字平均長度，字典的 key 的複雜度 (value 只有 包括 True 或 False 的結果，value 的複雜度為 1。)。
  - $N$ (系統堆疊空間)：遞迴的最大深度。每次遞迴刪除一個字母，最多遞迴深度到 $N$（直到剩 1 個字母）。

In [1]:
class Solution:
    def __init__(self, valid_words: set):
        # 初始化時將合法單字集存入，使用 set 以達到 O(1) 查詢
        self.valid_words = valid_words # space: O(M)
        # 記憶化字典：紀錄單字是否為 reducible，避免之後query重複計算
        self.memory = {}  # space: O(K * N)
    
    def is_reducible(self, word: str) -> bool: # time: O(K), space: O(N)
        # --- 邏輯區塊 1：基礎邊界處理 ---
        # 如果單字不在合法清單中，直接返回 False
        if word not in self.valid_words:
            print(f"{word} not in {self.valid_words} --> return False")
            return False
            
        # 如果長度為 1 且在合法清單中（上方已檢查），則為 True
        if len(word) == 1:
            print(f"len({word}) == 1 -> return True")
            return True
            
        # --- 邏輯區塊 2：檢查記憶化快取 ---
        # 如果這個單字之前算過了，直接回傳結果
        if word in self.memory:
            print(f"{word} in memo -> return {self.memory[word]}")
            return self.memory[word]
            
        # --- 邏輯區塊 3：遞迴嘗試刪除字母 ---
        # 遍歷單字的每個位置，嘗試刪除該字母
        for i in range(len(word)): # time: O(N)
            print("-" * 100)
            # 產生刪除第 i 個字母後的新字串
            current_word = word[:i] + word[i+1:] # time: O(N)
            print(f"{word = } -> remove index: {i}({word[i]}), {current_word = }")
            
            # 只要有一條路徑能成功縮減到底，目前這個單字就是 True
            status = self.is_reducible(current_word)
            print(f"{word = }, {current_word = }, {status = }")
            if status:
                self.memory[word] = True
                print(f"self.memo[{word}] = True -> {self.memory = }")
                return True
        
        
        # 如果所有路徑都試過了都不行，紀錄並返回 False
        self.memory[word] = False
        print(f"self.memo[{word}] = False -> {self.memory = }")
        print("_" * 100)
        return False

In [2]:
# 建立一個模擬的字典
valid_words = {"SPRINT", "PRINT", "PINT", "PIT", "IT", "I", "WITCH", "WITH", "WIT", "IT", "I"}
solver = Solution(valid_words)

target = "SPRINT"
print(f"單字 '{target}' 是否可縮減？ {solver.is_reducible(target)}")

print("=" * 100)

target2 = "WITCH"
print(f"單字 '{target2}' 是否可縮減？ {solver.is_reducible(target2)}")

----------------------------------------------------------------------------------------------------
word = 'SPRINT' -> remove index: 0(S), current_word = 'PRINT'
----------------------------------------------------------------------------------------------------
word = 'PRINT' -> remove index: 0(P), current_word = 'RINT'
RINT not in {'PIT', 'I', 'PINT', 'WITCH', 'IT', 'SPRINT', 'WITH', 'PRINT', 'WIT'} --> return False
word = 'PRINT', current_word = 'RINT', status = False
----------------------------------------------------------------------------------------------------
word = 'PRINT' -> remove index: 1(R), current_word = 'PINT'
----------------------------------------------------------------------------------------------------
word = 'PINT' -> remove index: 0(P), current_word = 'INT'
INT not in {'PIT', 'I', 'PINT', 'WITCH', 'IT', 'SPRINT', 'WITH', 'PRINT', 'WIT'} --> return False
word = 'PINT', current_word = 'INT', status = False
-----------------------------------------------------